In [8]:
# ============================================================
# Cell 1: Load Silver Data
# Source: silver_globalwatch lakehouse (attached in Explorer)
# Purpose: Load cleaned, validated air quality readings
# ============================================================

from pyspark.sql.functions import col, when, first
from pyspark.sql import functions as F

# Read silver_readings Delta table via Spark SQL
# Using SQL instead of abfss path since lakehouse is attached
df = spark.sql("SELECT * FROM silver_readings")

# Validate load
print(f"Total rows: {df.count()}")
df.printSchema()
df.show(5)

StatementMeta(, a1966e5f-838d-4ea8-8d21-3f9326ef9d6d, 21, Finished, Available, Finished, False)

Total rows: 344
root
 |-- location_id: integer (nullable = true)
 |-- location_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- country_name: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- parameter: string (nullable = true)
 |-- value: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- aqi_category: string (nullable = true)
 |-- reading_ts: timestamp (nullable = true)
 |-- reading_date: date (nullable = true)
 |-- reading_hour: integer (nullable = true)
 |-- year_month: string (nullable = true)
 |-- is_recent: boolean (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- silver_processed_ts: timestamp (nullable = true)
 |-- source_system: string (nullable = true)

+-----------+--------------------+------+------------+--------------+-----------------+-------------------+---------+-----+-----+------------+-------------------+--

In [9]:
# ============================================================
# Cell 2: Exploratory Data Analysis (EDA)
# Purpose: Understand parameter and AQI distribution
#          before feature engineering
# ============================================================

# Count of readings per pollutant type
# Helps understand data balance across parameters
print("=== Parameter distribution ===")
df.groupBy("parameter").count().orderBy("count", ascending=False).show()

# Count of AQI categories
# High N/A count expected — only pm25 has WHO thresholds defined
print("=== AQI Category distribution ===")
df.groupBy("aqi_category").count().orderBy("count", ascending=False).show()

# Descriptive stats for PM2.5 values
# Min/Max/StdDev helps detect outliers before training
print("=== PM25 value stats ===")
df.filter(col("parameter") == "pm25").select("value").describe().show()

StatementMeta(, a1966e5f-838d-4ea8-8d21-3f9326ef9d6d, 23, Finished, Available, Finished, False)

=== Parameter distribution ===
+---------+-----+
|parameter|count|
+---------+-----+
|       o3|   86|
|      no2|   77|
|     pm25|   74|
|     pm10|   72|
|       co|   35|
+---------+-----+

=== AQI Category distribution ===
+------------+-----+
|aqi_category|count|
+------------+-----+
|         N/A|  270|
|        Good|   44|
|    Moderate|   19|
|   Unhealthy|    6|
|   Hazardous|    5|
+------------+-----+

=== PM25 value stats ===
+-------+------------------+
|summary|             value|
+-------+------------------+
|  count|                74|
|   mean| 32.92999999999999|
| stddev|59.318995825565224|
|    min|               1.0|
|    max|             300.0|
+-------+------------------+



In [10]:
# ============================================================
# Cell 3: Feature Engineering — Pivot + Label Creation
# Problem: Data is in long format (one row per parameter)
# Solution: Pivot to wide format (one row per station+timestamp)
#           so each pollutant becomes a feature column
# ============================================================

# Pivot long → wide: each parameter becomes its own column
# groupBy: station identity + timestamp (one reading event)
# pivot: creates pm25, pm10, no2, o3, co columns
# agg: take first value if duplicates exist
df_pivot = df.groupBy(
    "location_id", "location_name", "city",
    "country_code", "country_name", "reading_ts"
) \
    .pivot("parameter", ["pm25", "pm10", "no2", "o3", "co"]) \
    .agg(first("value"))

# Replace nulls with 0 — station may not measure all parameters
# e.g. a station measuring only pm25 will have 0 for no2, o3, co
df_pivot = df_pivot.fillna(0)

# Create classification label based on WHO PM2.5 thresholds
# This is our target variable (what we want the model to predict)
# 0 = Good       (pm25 <= 12 µg/m³)
# 1 = Moderate   (pm25 <= 35 µg/m³)
# 2 = Unhealthy for Sensitive Groups (pm25 <= 55 µg/m³)
# 3 = Unhealthy  (pm25 <= 150 µg/m³)
# 4 = Hazardous  (pm25 > 150 µg/m³)
df_pivot = df_pivot.withColumn("aqi_label",
    when(col("pm25") <= 12, 0)
    .when(col("pm25") <= 35, 1)
    .when(col("pm25") <= 55, 2)
    .when(col("pm25") <= 150, 3)
    .otherwise(4)
)

# Validate pivot output
print(f"Pivoted rows: {df_pivot.count()}")
print("\n=== AQI Label distribution (0=Good → 4=Hazardous) ===")
df_pivot.groupBy("aqi_label").count().orderBy("aqi_label").show()
df_pivot.show(5)

StatementMeta(, a1966e5f-838d-4ea8-8d21-3f9326ef9d6d, 25, Finished, Available, Finished, False)

Pivoted rows: 164

=== AQI Label distribution (0=Good → 4=Hazardous) ===
+---------+-----+
|aqi_label|count|
+---------+-----+
|        0|  134|
|        1|   19|
|        3|    6|
|        4|    5|
+---------+-----+

+-----------+--------------------+--------------------+------------+--------------+-------------------+----+----+----+----+---+---------+
|location_id|       location_name|                city|country_code|  country_name|         reading_ts|pm25|pm10| no2|  o3| co|aqi_label|
+-----------+--------------------+--------------------+------------+--------------+-------------------+----+----+----+----+---+---------+
|        200|     Meyer Park C561|       United States|          US| United States|2016-01-30 00:00:00| 0.0| 0.0| 0.0|0.03|0.0|        0|
|        171|              U of H|       United States|          US| United States|2016-01-30 01:00:00| 0.0| 0.0| 0.0|0.04|0.0|        0|
|        137|   Harrow - Stanmore|              Harrow|          GB|United Kingdom|2016-01-3

In [11]:
# ============================================================
# Cell 4: Model Training — Random Forest Classifier
# Algorithm: Random Forest (ensemble of decision trees)
# Why RF: Handles class imbalance well, no feature scaling needed,
#         gives feature importances, robust on small datasets
# Tracking: MLflow — logs params, metrics, registers model
# ============================================================

import mlflow
import mlflow.spark
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

# Feature columns — all 5 air quality pollutants
# Model learns which combination of pollutants predicts AQI class
feature_cols = ["pm25", "pm10", "no2", "o3", "co"]

# VectorAssembler: combines individual feature columns into
# a single dense vector — required input format for Spark ML
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# 80/20 split — seed=42 ensures reproducible splits across runs
train_df, test_df = df_pivot.randomSplit([0.8, 0.2], seed=42)
print(f"Train rows: {train_df.count()}, Test rows: {test_df.count()}")

# Set MLflow experiment — groups all runs under one experiment name
# Visible in Fabric ML Experiments section
mlflow.set_experiment("globalwatch_aqi_prediction")

with mlflow.start_run(run_name="random_forest_aqi_v1"):

    # Random Forest Classifier
    # numTrees=100: more trees = more stable predictions
    # maxDepth=5: limits overfitting on small dataset (164 rows)
    # seed=42: reproducible model
    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol="aqi_label",
        numTrees=100,
        maxDepth=5,
        seed=42
    )

    # Pipeline: chains assembler → classifier
    # Ensures same transformations apply at inference time
    pipeline = Pipeline(stages=[assembler, rf])

    # Train model on 80% training data
    model = pipeline.fit(train_df)

    # Generate predictions on 20% held-out test set
    predictions = model.transform(test_df)

    # Evaluate accuracy — ratio of correct predictions
    evaluator = MulticlassClassificationEvaluator(
        labelCol="aqi_label",
        predictionCol="prediction",
        metricName="accuracy"
    )
    accuracy = evaluator.evaluate(predictions)

    # Log hyperparameters to MLflow
    # Enables comparison across runs in MLflow UI
    mlflow.log_param("num_trees", 100)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("features", str(feature_cols))
    mlflow.log_param("train_size", "80%")
    mlflow.log_param("split_seed", 42)

    # Log evaluation metrics to MLflow
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("train_rows", train_df.count())
    mlflow.log_metric("test_rows", test_df.count())

    # Log model artifact + register in MLflow Model Registry
    # registered_model_name: creates versioned model entry
    # Version 1 auto-created on first registration
    mlflow.spark.log_model(
        model,
        "aqi_rf_model",
        registered_model_name="globalwatch_aqi_classifier"
    )

    print(f"✅ Accuracy: {accuracy:.4f}")
    print(f"✅ Model registered: globalwatch_aqi_classifier")

    # Spot-check predictions vs actual labels
    predictions.select(
        "pm25", "pm10", "no2",
        "aqi_label",      # actual
        "prediction"      # model predicted
    ).show(10)

StatementMeta(, a1966e5f-838d-4ea8-8d21-3f9326ef9d6d, 27, Finished, Available, Finished, False)

Train rows: 138, Test rows: 26
✅ Accuracy: 0.9615
✅ Model registered: globalwatch_aqi_classifier
+-----+-----+----+---------+----------+
| pm25| pm10| no2|aqi_label|prediction|
+-----+-----+----+---------+----------+
|110.0|182.0|41.3|        3|       3.0|
|134.0|179.0| 0.0|        3|       3.0|
|  0.0| 26.8| 0.0|        0|       0.0|
| 13.1| 23.5|14.0|        1|       1.0|
| 15.5| 20.2|10.2|        1|       1.0|
| 7.36|  0.0| 0.0|        0|       0.0|
| 34.0|  0.0| 0.0|        1|       1.0|
|  0.0| 12.3| 0.0|        0|       0.0|
|  0.0|  0.0| 0.0|        0|       0.0|
|  0.0|  0.0| 0.0|        0|       0.0|
+-----+-----+----+---------+----------+
only showing top 10 rows



2026/08/09 09:40:34 INFO mlflow.tracking.fluent: Experiment with name 'globalwatch_aqi_prediction' does not exist. Creating a new experiment.
2026/08/09 09:41:15 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp6n1dqo_i/model, flavor: spark). Fall back to return ['pyspark==3.5.1.5.4.20240407']. Set logging level to DEBUG to see the full traceback. 
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
Successfully registered model 'globalwatch_aqi_classifier'.


In [12]:
# ============================================================
# Cell 5: Feature Importance Analysis
# Purpose: Understand which pollutant drives AQI prediction
# Interview point: Shows model interpretability awareness
# ============================================================

# Extract the RandomForest model from the pipeline stages
# stages[-1] = last stage = the classifier (after assembler)
rf_model = model.stages[-1]

# featureImportances: Gini impurity-based importance scores
# Higher score = pollutant contributes more to AQI classification
importances = rf_model.featureImportances

# Zip feature names with their importance scores
feature_importance = list(zip(feature_cols, importances))

# Sort descending — most important pollutant first
feature_importance.sort(key=lambda x: x[1], reverse=True)

print("=== Feature Importances (higher = more predictive) ===")
for feature, importance in feature_importance:
    bar = "█" * int(importance * 50)  # visual bar
    print(f"  {feature:>5}: {importance:.4f}  {bar}")

StatementMeta(, a1966e5f-838d-4ea8-8d21-3f9326ef9d6d, 28, Finished, Available, Finished, False)

=== Feature Importances (higher = more predictive) ===
   pm25: 0.7632  ██████████████████████████████████████
   pm10: 0.1291  ██████
    no2: 0.0469  ██
     co: 0.0378  █
     o3: 0.0230  █


In [16]:
# ============================================================
# Cell 6: Apply Trained Model to Gold fact_readings
# Purpose: Enrich Gold table with ML-predicted AQI class
# Pattern: Load registered model → transform → write back
# ============================================================

import mlflow.spark

# Load registered model from MLflow Model Registry
loaded_model = mlflow.spark.load_model(
    "models:/globalwatch_aqi_classifier/1"
)

# Load gold fact_readings
gold_df = spark.sql("SELECT * FROM gold_globalwatch.dbo.fact_readings")

# Filter to pm25 readings and reshape for model input
# Keep location context columns alongside feature columns
pm25_df = gold_df.filter(col("parameter") == "pm25") \
    .withColumn("pm25", col("value")) \
    .withColumn("pm10", F.lit(0.0)) \
    .withColumn("no2", F.lit(0.0)) \
    .withColumn("o3", F.lit(0.0)) \
    .withColumn("co", F.lit(0.0))

# Run model inference
predictions = loaded_model.transform(pm25_df)

# Show sample predictions with location context
predictions.select(
    "location_id",
    "country_sk",
    "pm25",
    "prediction"    # 0=Good, 1=Moderate, 2=USG, 3=Unhealthy, 4=Hazardous
).show(10)

print(f"✅ Predictions generated: {predictions.count()} rows")

StatementMeta(, a1966e5f-838d-4ea8-8d21-3f9326ef9d6d, 32, Finished, Available, Finished, False)

2026/08/09 09:47:44 INFO mlflow.spark: 'models:/globalwatch_aqi_classifier/1' resolved as 'abfss://5b66cff3-e0eb-45a0-8553-7ea97a757336@onelakecentralindia.pbidedicated.windows.net/72990e46-1883-49f2-8400-793c151de348/Data/4d55a91b-2ee5-49ff-92c8-ca4a7294db10/artifacts'


2026/08/09 09:47:44 INFO mlflow.store.artifact.artifact_repo: The progress bar can be disabled by setting the environment variable MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR to false
2026/08/09 09:47:45 INFO mlflow.spark: File 'models:/globalwatch_aqi_classifier/1/sparkml' not found on DFS. Will attempt to upload the file.
2026/08/09 09:47:46 INFO mlflow.spark: Copied SparkML model to Files/tmp/mlflow/f56ac29b-b000-4aaf-abd8-73a1a350600d


+-----------+-----------+-----+----------+
|location_id| country_sk| pm25|prediction|
+-----------+-----------+-----+----------+
|         28|-2079833584| 13.1|       1.0|
|         43|-2079833584| 15.5|       1.0|
|         21|  199844526| 10.0|       0.0|
|        142|  199844526| 89.0|       3.0|
|        143|  199844526| 24.0|       1.0|
|        144|  199844526| 18.0|       1.0|
|         19|-1788195040|172.0|       3.0|
|         46|-1788195040|207.0|       4.0|
|         48|-1788195040|114.0|       3.0|
|         31| 1346993615| 3.76|       0.0|
+-----------+-----------+-----+----------+
only showing top 10 rows

✅ Predictions generated: 74 rows


StatementMeta(, a1966e5f-838d-4ea8-8d21-3f9326ef9d6d, 33, Finished, Available, Finished, False)

In [17]:
# -------------------------------------------------------
# CELL 7: Write ML Predictions to Gold Lakehouse
# Purpose: Persist predicted AQI classes as a new Delta
#          table in Gold layer for downstream consumption
# -------------------------------------------------------

from pyspark.sql.functions import col, when

# Map numeric prediction back to human-readable AQI class label
# Matches the WHO threshold labels used during feature engineering
predictions_final = predictions.select(
    "location_id",
    "country_sk",
    "pollutant_sk",
    "date_key",
    "pm25",
    "prediction"
).withColumn("predicted_aqi_class",
    when(col("prediction") == 0, "Good")
    .when(col("prediction") == 1, "Moderate")
    .when(col("prediction") == 2, "Unhealthy for Sensitive")
    .when(col("prediction") == 3, "Unhealthy")
    .otherwise("Hazardous")
)

# Persist predictions as a new Delta table in Gold lakehouse
# This enriches the Gold layer with ML-derived AQI classifications
# mode=overwrite: safe to re-run — always reflects latest model version
predictions_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_globalwatch.dbo.fact_aqi_predictions")

print(f"✅ Written to gold_globalwatch.dbo.fact_aqi_predictions")

# Validate distribution — confirms model is not predicting single class
predictions_final.groupBy("predicted_aqi_class") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

StatementMeta(, a1966e5f-838d-4ea8-8d21-3f9326ef9d6d, 34, Finished, Available, Finished, False)

✅ Written to gold_globalwatch.dbo.fact_aqi_predictions
+-------------------+-----+
|predicted_aqi_class|count|
+-------------------+-----+
|               Good|   42|
|           Moderate|   21|
|          Unhealthy|    7|
|          Hazardous|    4|
+-------------------+-----+

